In [1]:
"""Split a patent sheet into its separate figures.   (v2)

Rule of thumb: never merge two things just because their bounding *boxes* touch.

  1. work on the ink (connected components), not on filled rectangles
  2. SEEDS = big components. Seeds merge only if their *ink* nearly touches, or one sits
     in the other's box AND is still close to its ink. A figure that merely sits in the
     whitespace of a bigger figure's box (an inset like Fig 4) stays a figure of its own.
  3. every smaller piece (label letter, arrow, lever, dash of a section line) joins the
     NEAREST figure if its ink is within ATTACH px of that figure's ink. This chains, so
     a letter can join through its leader line or through the dashes of a section line.
     Pieces that never get close to a seed (title, footer, captions, dust) are left out.
  4. a figure is whatever contains at least one seed. There is no page-area threshold,
     so small figures (a Fig 3 strip, a Fig 4 inset) survive.

All distances below are in pixels at 300 dpi (A4 = 3509 x 2480). For another scan
resolution pass dpi=... to find_figures. It is explicit on purpose: stitched pages change
the image size, so the size alone says nothing about the resolution.
"""
import cv2 as cv
import numpy as np

# --- tunables (px @ 300 dpi) -------------------------------------------------------
MIN_AREA  = 20     # ink pixels; drops speckles
SEED_AREA = 3000   # ink pixels; components at least this big are figure seeds
TOUCH     = 8      # seeds closer than this (ink to ink) are one figure (broken strokes, levers)
NEAR      = 45     # ... or closer than this if one lies inside the other's box (CONTAIN)
CONTAIN   = 0.5    # share of the smaller box that must lie inside the bigger box
ATTACH    = 48     # small pieces join the nearest figure within this distance, chaining
                   # Measured on the sample sheets: label pieces end up <= 46 px from their
                   # figure, captions / printed title start at >= 53 px. Raise ATTACH to
                   # also pull in captions, lower it if labels leak onto the next figure.
                   # Different figures are >= 19 px apart, so keep TOUCH well below that.


def area(b):
    return (b[2] - b[0]) * (b[3] - b[1])


def frac_inside(piece, host):
    """Share of `piece`'s box lying inside `host`'s box (overlap / piece area, not IoU)."""
    iw = min(piece[2], host[2]) - max(piece[0], host[0])
    ih = min(piece[3], host[3]) - max(piece[1], host[1])
    return max(iw, 0) * max(ih, 0) / area(piece)


def _dist_to(mask):
    """Distance of every pixel to the nearest True pixel of `mask`."""
    return cv.distanceTransform(np.where(mask, 0, 1).astype(np.uint8), cv.DIST_L2, 5)


def _union(a, b):
    return [min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3])]


def find_figures(ink, min_area=MIN_AREA, seed_area=SEED_AREA, touch=TOUCH, near=NEAR,
                 contain=CONTAIN, attach=ATTACH, dpi=300):
    """ink: uint8 image, ink = 255. Returns a list of [x1, y1, x2, y2] (x2, y2 exclusive)."""
    H, W = ink.shape
    s = dpi / 300.0                                           # px constants are for 300 dpi
    min_area, seed_area = min_area * s * s, seed_area * s * s
    touch, near, attach = touch * s, near * s, attach * s

    n, lab, st, _ = cv.connectedComponentsWithStats(ink, connectivity=8)
    x, y, w, h, a = (st[:, k].astype(np.int64) for k in range(5))
    box = np.stack([x, y, x + w, y + h], 1)
    fill = a / np.maximum(w * h, 1)
    aspect = np.maximum(w, h) / np.maximum(np.minimum(w, h), 1)

    big = a >= seed_area
    # scan artefacts: big but thin (border / margin strips) or solid (black blobs, wedges)
    junk = big & (((aspect >= 15) & (np.minimum(w, h) < 0.03 * W)) | (fill > 0.8))
    keep = (a >= min_area) & ~junk
    keep[0] = False
    seeds = [i for i in np.argsort(-a) if keep[i] and big[i]]
    pieces = {i for i in range(1, n) if keep[i] and not big[i]}

    # ---- 1. seeds -> figures ---------------------------------------------------------
    groups = [dict(ids=[i], box=list(box[i]), ink=int(a[i])) for i in seeds]

    def mask_of(g, roi):
        return np.isin(lab[roi[1]:roi[3], roi[0]:roi[2]], g["ids"])

    def ink_gap(g1, g2):
        b1, b2 = g1["box"], g2["box"]
        if np.hypot(max(b1[0] - b2[2], b2[0] - b1[2], 0),
                    max(b1[1] - b2[3], b2[1] - b1[3], 0)) > near:
            return np.inf                                     # boxes too far apart to bother
        roi = _union(b1, b2)
        return float(_dist_to(mask_of(g1, roi))[mask_of(g2, roi)].min())

    merged = True
    while merged:
        merged = False
        for p in range(len(groups)):
            for q in range(p + 1, len(groups)):
                host, small = sorted((groups[p], groups[q]), key=lambda g: -g["ink"])
                g = ink_gap(host, small)
                if g <= touch or (g <= near and frac_inside(small["box"], host["box"]) >= contain):
                    host["ids"] += small["ids"]
                    host["box"] = _union(host["box"], small["box"])
                    host["ink"] += small["ink"]
                    groups.remove(small)
                    merged = True
                    break
            if merged:
                break

    # ---- 2. small pieces -> nearest figure, chaining through attached pieces ----------
    for _ in range(8):
        claims = {}                                           # piece -> (distance, group)
        for gi, g in enumerate(groups):
            x1, y1 = max(int(g["box"][0] - attach), 0), max(int(g["box"][1] - attach), 0)
            x2, y2 = min(int(g["box"][2] + attach) + 1, W), min(int(g["box"][3] + attach) + 1, H)
            roi_lab, dt = lab[y1:y2, x1:x2], None
            for i in pieces:
                bx = box[i]
                if bx[2] < x1 or bx[0] > x2 or bx[3] < y1 or bx[1] > y2:
                    continue
                if dt is None:
                    dt = _dist_to(np.isin(roi_lab, g["ids"]))
                ys = slice(max(bx[1], y1) - y1, min(bx[3], y2) - y1)
                xs = slice(max(bx[0], x1) - x1, min(bx[2], x2) - x1)
                own = roi_lab[ys, xs] == i
                if own.any():
                    d = float(dt[ys, xs][own].min())
                    if d <= attach and d < claims.get(i, (np.inf,))[0]:
                        claims[i] = (d, gi)
        if not claims:
            break
        for i, (_, gi) in claims.items():
            groups[gi]["ids"].append(i)
            groups[gi]["box"] = _union(groups[gi]["box"], box[i])
            pieces.discard(i)

    return [[int(v) for v in g["box"]] for g in groups]


def reading_order(boxes):
    """Rows top to bottom (boxes sharing >= 50% of their height form a row), left to right inside a row."""
    rows = []
    for b in sorted(boxes, key=lambda b: b[1]):
        for r in rows:
            top, bot = min(c[1] for c in r), max(c[3] for c in r)
            if min(bot, b[3]) - max(top, b[1]) >= 0.5 * min(bot - top, b[3] - b[1]):
                r.append(b)
                break
        else:
            rows.append([b])
    return [b for r in rows for b in sorted(r, key=lambda b: b[0])]


In [3]:
import glob
from pathlib import Path as PATH
import os


def ink_based_connected_components(src, debug_out):
    gray = cv.imread(src, cv.IMREAD_GRAYSCALE)
    _, ink = cv.threshold(gray, 128, 255, cv.THRESH_BINARY_INV)
    figures = reading_order(find_figures(ink))
    debug = cv.cvtColor(ink, cv.COLOR_GRAY2BGR)
    for i, (x1, y1, x2, y2) in enumerate(figures, 1):
        cv.rectangle(debug, (x1, y1), (x2 - 1, y2 - 1), (0, 255, 0), 2)   # corners are inclusive
        cv.putText(debug, str(i), (x1 + 8, y1 + 40), cv.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3)
    # uncomment the following lines to save each figure as a separate image
    #     cv.imwrite(f"figure_{i}.png", gray[y1:y2, x1:x2])
    cv.imwrite(debug_out, debug)
    # print(f"{len(figures)} figures:", figures)


directory_path = "../../data/figure_extractor_test/"
all_sketch_pages = glob.glob(directory_path + "*.png")
debug_directory = PATH("../../results/figure_extractor_v2_test_output/")
debug_directory.mkdir(parents=True, exist_ok=True)

for src in all_sketch_pages:
    src_filename = src.split("/")[-1]
    debug_out = os.path.join(debug_directory, PATH(f"debug_{src_filename}"))
    print(f"Debug output saved to: {debug_out}")
    ink_based_connected_components(src, debug_out)


Debug output saved to: ../../results/figure_extractor_v2_test_output/debug_2393_2398_page_3.png
Debug output saved to: ../../results/figure_extractor_v2_test_output/debug_2424_2429_page_2.png
Debug output saved to: ../../results/figure_extractor_v2_test_output/debug_2424_2429_page_3.png
Debug output saved to: ../../results/figure_extractor_v2_test_output/debug_2398_2403_page_3.png
Debug output saved to: ../../results/figure_extractor_v2_test_output/debug_sheet_pages_0003_0005.png
Debug output saved to: ../../results/figure_extractor_v2_test_output/debug_2398_2403_page_4.png
Debug output saved to: ../../results/figure_extractor_v2_test_output/debug_sheet_pages_0010_0012.png
Debug output saved to: ../../results/figure_extractor_v2_test_output/debug_sheet_pages_0008_0009.png
Debug output saved to: ../../results/figure_extractor_v2_test_output/debug_2453_2458_page_3.png
Debug output saved to: ../../results/figure_extractor_v2_test_output/debug_2394_2399_page_2.png
Debug output saved to: ..